<img src="./static/imo_health.png" alt="IMO Health Logo" width="300"/>

---

# IMO Knowledge Graph Traversal

This notebook demonstrates querying the IMO Knowledge Graph GraphQL endpoint and formatting mappings in a table.

Endpoint: `https://api.imohealth.com/knowledgegraph/graphql/`

## Step 1: Install Packages and Load Configuration

Copy `config.json.template` to `config.json` and provide IMO client credentials.

In [ ]:
%pip install requests pandas --quiet

import json
import pathlib
import requests
import pandas as pd
from IPython.display import display

candidates = [
    pathlib.Path('config.json'),
    pathlib.Path.cwd() / 'config.json',
    pathlib.Path.cwd().parent / 'config.json'
]
cfg_path = next((p for p in candidates if p.exists()), None)

if cfg_path is None:
    raise FileNotFoundError('config.json not found. Copy config.json.template to config.json and fill in credentials.')

with open(cfg_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

kg_cfg = cfg.get('knowledge_graph', {})
CLIENT_ID = kg_cfg.get('client_id', '')
CLIENT_SECRET = kg_cfg.get('client_secret', '')
TOKEN_URL = kg_cfg.get('token_url', 'https://api.imohealth.com/oauth/token')
GRAPHQL_URL = kg_cfg.get('graphql_url', 'https://api.imohealth.com/knowledgegraph/graphql/')

print(f'Loaded config from: {cfg_path.resolve()}')
print('client_id configured:', bool(CLIENT_ID and CLIENT_ID != '<YOUR_KNOWLEDGE_GRAPH_CLIENT_ID>'))
print('client_secret configured:', bool(CLIENT_SECRET and CLIENT_SECRET != '<YOUR_KNOWLEDGE_GRAPH_CLIENT_SECRET>'))

## Step 2: Get OAuth Access Token

In [ ]:
def get_token(client_id: str, client_secret: str) -> str:
    payload = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'audience': 'https://api.imohealth.com'
    }
    resp = requests.post(TOKEN_URL, json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()['access_token']

access_token = get_token(CLIENT_ID, CLIENT_SECRET)
print('Token acquired (prefix):', access_token[:20] + '...')

## Step 3: Query the Mappings in Knowledge Graph 

First query: retrieve code system mappings for IMO lexical code `85191`.

> **Schema note:** `mappings` is defined on `ProblemLexical` (a concrete subtype), not on the base `Lexical` interface. Use an inline fragment `... on ProblemLexical` to access it.

In [ ]:
graphql_query = '''
query get_mappings{
  lexical(code: "85191") {
    title
    ... on ProblemLexical {
      mappings {
        code
        codeSystem
      }
    }
  }
}
'''

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json'
}

response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': graphql_query},
    timeout=60
)
response.raise_for_status()
result = response.json()

lexical = result.get('data', {}).get('lexical', {})
title = lexical.get('title', '')
mappings = lexical.get('mappings', [])

print('Lexical title:', title)
df = pd.DataFrame(mappings)

styled_df = df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
    {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
    {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', '60%')]} 
])
display(styled_df)

print('Raw JSON response:')
print(json.dumps(result, indent=2))

## Step 4: Navigate Concept Hierarchy

Run a hierarchy query for lexical code `85191` and display `domainBroader`/`domainNarrower` concepts in pretty tables.

> **Schema note:** `ProblemLexical` uses `domainBroader` and `domainNarrower` (domain hierarchy fields) instead of the generic `broader`/`narrower`.

In [ ]:
hierarchy_query = '''
query get_hierachy{
  lexical(code: "85191") {
    title
    ... on ProblemLexical {
      domainBroader {
        code
        title
        ... on ProblemLexical {
          mappings {
            code
            codeSystem
          }
        }
      }
      domainNarrower {
        code
        title
      }
    }
  }
}
'''

hierarchy_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': hierarchy_query},
    timeout=60
)
hierarchy_response.raise_for_status()
hierarchy_result = hierarchy_response.json()

hier_lexical = hierarchy_result.get('data', {}).get('lexical', {})
hier_title = hier_lexical.get('title', '')
domain_broader = hier_lexical.get('domainBroader', [])
domain_narrower = hier_lexical.get('domainNarrower', [])

print('Lexical title:', hier_title)

def purple_style(df: pd.DataFrame, width: str = '80%'):
    return df.style.set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
    ])

# Domain broader concepts table
broader_df = pd.DataFrame([{'code': b.get('code', ''), 'title': b.get('title', '')} for b in domain_broader])
print('\nDomain Broader Concepts')
display(purple_style(broader_df, width='55%'))

# Domain broader mappings table (flattened)
broader_mappings_rows = []
for b in domain_broader:
    for m in b.get('mappings', []):
        broader_mappings_rows.append({
            'broader_code': b.get('code', ''),
            'broader_title': b.get('title', ''),
            'mapping_code': m.get('code', ''),
            'code_system': m.get('codeSystem', '')
        })

broader_mappings_df = pd.DataFrame(broader_mappings_rows)
print('\nDomain Broader Mappings')
display(purple_style(broader_mappings_df, width='100%'))

# Domain narrower concepts table
narrower_df = pd.DataFrame([{'code': n.get('code', ''), 'title': n.get('title', '')} for n in domain_narrower])
print('\nDomain Narrower Concepts')
display(purple_style(narrower_df, width='100%'))

print('\nRaw JSON response:')
print(json.dumps(hierarchy_result, indent=2))

## Step 5: Drill Down Narrower Relationships

Run a second-level hierarchy query using `domainNarrower` and show parent-to-child relationships in a pretty table.

In [ ]:
drill_down_query = '''
query get_drill_down_hierarchy{
  lexical(code: "85191") {
    title
    ... on ProblemLexical {
      domainNarrower {
        code
        title
        ... on ProblemLexical {
          domainNarrower {
            code
            title
          }
        }
      }
    }
  }
}
'''

drill_down_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': drill_down_query},
    timeout=60
)
drill_down_response.raise_for_status()
drill_down_result = drill_down_response.json()

drill_lexical = drill_down_result.get('data', {}).get('lexical', {})
drill_title = drill_lexical.get('title', '')
level1_narrower = drill_lexical.get('domainNarrower', [])

print('Lexical title:', drill_title)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

# Level 1 domainNarrower summary
level1_df = pd.DataFrame([
    {
        'level1_code': item.get('code', ''),
        'level1_title': item.get('title', ''),
        'child_count': len(item.get('domainNarrower', []))
    }
    for item in level1_narrower
])
print('\nLevel 1 Domain Narrower Concepts (with child counts)')
display(style_fn(level1_df, width='100%'))

# Flatten level1 -> level2 relationships
drill_rows = []
for parent in level1_narrower:
    parent_code = parent.get('code', '')
    parent_title = parent.get('title', '')
    children = parent.get('domainNarrower', [])

    if children:
        for child in children:
            drill_rows.append({
                'parent_code': parent_code,
                'parent_title': parent_title,
                'child_code': child.get('code', ''),
                'child_title': child.get('title', '')
            })
    else:
        drill_rows.append({
            'parent_code': parent_code,
            'parent_title': parent_title,
            'child_code': '',
            'child_title': ''
        })

drill_df = pd.DataFrame(drill_rows)
print('\nDrill-Down Domain Narrower Relationships (Level 1 -> Level 2)')
display(style_fn(drill_df, width='100%'))

print('\nRaw JSON response:')
print(json.dumps(drill_down_result, indent=2))

## Step 6: Query Allowed Refinements and Groups

Run a refinements query for lexical code `85191` and show each concept with its IMO Health refinement group.

Use the Knowledge graph content on Concept Refinements to build Refinement workflows.

> **Schema note:** `ProblemLexical` exposes `allowedRefinements` (via `RefinementHierarchy`) and `refinementNarrower(refinements: [...])` for filtered navigation.

In [ ]:
refinements_query = '''
query get_refinements{
  lexical(code: "85191") {
    title
    ... on ProblemLexical {
      allowedRefinements{
        code
        title
        group {
          code
          title
        }
      }
    }
  }
}
'''

refinements_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': refinements_query},
    timeout=60
)
refinements_response.raise_for_status()
refinements_result = refinements_response.json()

ref_lexical = refinements_result.get('data', {}).get('lexical', {})
ref_title = ref_lexical.get('title', '')
allowed_refinements = ref_lexical.get('allowedRefinements', [])

print('Lexical title:', ref_title)

refinement_rows = []
for r in allowed_refinements:
    g = r.get('group', {})
    refinement_rows.append({
        'concept_code': r.get('code', ''),
        'concept_title': r.get('title', ''),
        'group_code': g.get('code', ''),
        'group_title': g.get('title', '')
    })

refinements_df = pd.DataFrame(refinement_rows)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

print('\nRefinement Concepts with Groups')
display(style_fn(refinements_df, width='100%'))

group_summary_df = (
    refinements_df.groupby(['group_code', 'group_title'], dropna=False)
    .size()
    .reset_index(name='concept_count')
    .sort_values(by='concept_count', ascending=False)
    .reset_index(drop=True)
)

print('\nRefinement Group Summary')
display(style_fn(group_summary_df, width='70%'))

print('\nRaw JSON response:')
print(json.dumps(refinements_result, indent=2))

## Step 7: Sequential Refinements

Apply refinements sequentially — each `refinementNarrower` result is further narrowed by the next refinement code in the chain:

`75952` → narrow by `1403` → narrow by `1105` → narrow by `1112`

Each nested level needs `... on ProblemLexical` because `refinementNarrower` returns `[Lexical]`.

In [ ]:
sequential_refinements_query = '''
query get_sequential_refinements{
  lexical(code: "75952") {
    title
    ... on ProblemLexical {
      appliedRefinements {
        code
        title
        group {
          code
          title
        }
      }
      allowedRefinements {
        code
        title
        group {
          code
          title
        }
      }
      refinementNarrower(refinements: ["1403"]) {
        code
        title
        ... on ProblemLexical {
          appliedRefinements {
            code
            title
            group {
              code
              title
            }
          }
          allowedRefinements {
            code
            title
            group {
              code
              title
            }
          }
          refinementNarrower(refinements: ["1105"]) {
            code
            title
            ... on ProblemLexical {
              appliedRefinements {
                code
                title
                group {
                  code
                  title
                }
              }
              allowedRefinements {
                code
                title
                group {
                  code
                  title
                }
              }
              refinementNarrower(refinements: ["1112"]) {
                code
                title
              }
            }
          }
        }
      }
    }
  }
}
'''

seq_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': sequential_refinements_query},
    timeout=60
)
seq_response.raise_for_status()
seq_result = seq_response.json()

seq_lexical = seq_result.get('data', {}).get('lexical', {})
seq_title = seq_lexical.get('title', '')
level1 = seq_lexical.get('refinementNarrower', [])

print('Lexical title:', seq_title)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

def refinements_to_df(refinements):
    return pd.DataFrame([
        {
            'code': r.get('code', ''),
            'title': r.get('title', ''),
            'group_code': r.get('group', {}).get('code', ''),
            'group_title': r.get('group', {}).get('title', '')
        }
        for r in refinements
    ])

# Root level
print('\n--- Root (75952) ---')
print('Applied Refinements')
display(style_fn(refinements_to_df(seq_lexical.get('appliedRefinements', [])), width='100%'))
print('Allowed Refinements')
display(style_fn(refinements_to_df(seq_lexical.get('allowedRefinements', [])), width='100%'))

# Level 1 (1403)
for l1 in level1:
    print(f'\n--- After refinement 1403: {l1.get("code")} — {l1.get("title")} ---')
    print('Applied Refinements')
    display(style_fn(refinements_to_df(l1.get('appliedRefinements', [])), width='100%'))
    print('Allowed Refinements')
    display(style_fn(refinements_to_df(l1.get('allowedRefinements', [])), width='100%'))

    # Level 2 (1105)
    for l2 in l1.get('refinementNarrower', []):
        print(f'\n--- After refinement 1105: {l2.get("code")} — {l2.get("title")} ---')
        print('Applied Refinements')
        display(style_fn(refinements_to_df(l2.get('appliedRefinements', [])), width='100%'))
        print('Allowed Refinements')
        display(style_fn(refinements_to_df(l2.get('allowedRefinements', [])), width='100%'))

        # Level 3 (1112) — final results
        final = l2.get('refinementNarrower', [])
        if final:
            final_df = pd.DataFrame([{'code': c.get('code', ''), 'title': c.get('title', '')} for c in final])
            print(f'\nFinal results after refinement 1112 ({len(final)} concepts)')
            display(style_fn(final_df, width='80%'))
        else:
            print('\nNo concepts matched refinement 1112.')

print('\nRaw JSON response:')
print(json.dumps(seq_result, indent=2))

## Step 8: Cross-Domain Relationships

Query cross-domain links between domains using the `MedicationLexical` and `ProcedureLexical` types.

**`MedicationLexical` cross-domain fields:**
- `treatedProblems` — problems for which this medication is listed as a treatment
- `causedProblems` — problems for which this medication is listed as a causative agent
- `contraindicatedProblems` — problems for which this medication is contraindicated
- `preventedProblems` — problems for which this medication is listed as a preventive agent

**`ProcedureLexical` cross-domain fields:**
- `associatedProblems` — problems that list this procedure as an associated procedure
- `interpretedFindings` — problems that interpret this procedure
- `causedProblems` — problems that list this procedure as a due-to cause

### Step 8a: Medication → Problem Cross-Domain Links

Query `treatedProblems`, `causedProblems`, `contraindicatedProblems`, and `preventedProblems` for a `MedicationLexical` concept.

In [ ]:
medication_cross_domain_query = '''
query get_medication_cross_domain {
  lexical(code: "114959", domain: medication) {
    title
    ... on MedicationLexical {
      mappings {
        code
        codeSystem
      }
      treatedProblems {
        code
        title
      }
      causedProblems {
        code
        title
      }
      contraindicatedProblems {
        code
        title
      }
      preventedProblems {
        code
        title
      }
    }
  }
}
'''

med_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': medication_cross_domain_query},
    timeout=60
)
med_response.raise_for_status()
med_result = med_response.json()

med_lexical = med_result.get('data', {}).get('lexical', {})
med_title = med_lexical.get('title', '')

print('Medication lexical title:', med_title)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

# Mappings table
mappings = med_lexical.get('mappings', [])
mappings_df = pd.DataFrame([{'code': m.get('code', ''), 'codeSystem': m.get('codeSystem', '')} for m in mappings])
print(f'\nCode Mappings ({len(mappings)})')
if not mappings_df.empty:
    display(style_fn(mappings_df, width='60%'))
else:
    print('  (none)')

# Cross-domain tables
for field_key, label in [
    ('treatedProblems', 'Treated Problems'),
    ('causedProblems', 'Caused Problems'),
    ('contraindicatedProblems', 'Contraindicated Problems'),
    ('preventedProblems', 'Prevented Problems'),
]:
    items = med_lexical.get(field_key, [])
    df = pd.DataFrame([{'code': i.get('code', ''), 'title': i.get('title', '')} for i in items])
    print(f'\n{label} ({len(items)})')
    if not df.empty:
        display(style_fn(df, width='80%'))
    else:
        print('  (none)')

print('\nRaw JSON response:')
print(json.dumps(med_result, indent=2))

### Step 8b: Procedure → Problem Cross-Domain Links

Query `associatedProblems`, `interpretedFindings`, and `causedProblems` for a `ProcedureLexical` concept.

In [ ]:
procedure_cross_domain_query = '''
query get_procedure_cross_domain {
  lexical(code: "10000") {
    title
    ... on ProcedureLexical {
      mappings {
        code
        codeSystem
      }
      associatedProblems {
        code
        title
      }
      interpretedFindings {
        code
        title
      }
      causedProblems {
        code
        title
      }
    }
  }
}
'''

proc_response = requests.post(
    GRAPHQL_URL,
    headers=headers,
    json={'query': procedure_cross_domain_query},
    timeout=60
)
proc_response.raise_for_status()
proc_result = proc_response.json()

proc_lexical = proc_result.get('data', {}).get('lexical', {})
proc_title = proc_lexical.get('title', '')

print('Procedure lexical title:', proc_title)

if 'purple_style' in globals():
    style_fn = purple_style
else:
    def style_fn(df: pd.DataFrame, width: str = '100%'):
        return df.style.set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#6A0DAD'), ('color', 'white'), ('font-weight', 'bold')]},
            {'selector': 'td', 'props': [('border', '1px solid #ddd')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('width', width)]}
        ])

# Mappings table
mappings = proc_lexical.get('mappings', [])
mappings_df = pd.DataFrame([{'code': m.get('code', ''), 'codeSystem': m.get('codeSystem', '')} for m in mappings])
print(f'\nCode Mappings ({len(mappings)})')
if not mappings_df.empty:
    display(style_fn(mappings_df, width='60%'))
else:
    print('  (none)')

# Cross-domain tables
for field_key, label in [
    ('associatedProblems', 'Associated Problems'),
    ('interpretedFindings', 'Interpreted Findings'),
    ('causedProblems', 'Caused Problems'),
]:
    items = proc_lexical.get(field_key, [])
    df = pd.DataFrame([{'code': i.get('code', ''), 'title': i.get('title', '')} for i in items])
    print(f'\n{label} ({len(items)})')
    if not df.empty:
        display(style_fn(df, width='80%'))
    else:
        print('  (none)')

print('\nRaw JSON response:')
print(json.dumps(proc_result, indent=2))